# Signal Subcarrier Heatmaps

Purpose: visualize per-subcarrier amplitude, phase, and rolling variance heatmaps from a deterministic synthetic CSI capture.

Run path: install research extras with `uv sync --extra research`, open this notebook, and choose Run All. The notebook is self-contained and does not require hardware recordings.

Fixture / simulated source: `inline_heatmap_capture` creates a 120-frame, 3-stream, 64-subcarrier capture with static channel structure plus a localized motion event. A guarded import hook is present for future `ruview.signal.visualization` helpers, but the inline fixture is the default fallback.


In [ ]:
# Data source option: synthetic fixture or local ESP32 CSI recording.
USE_RECORDING = False
RECORDING_CSV = None  # Set to a specific *_csi.csv path, or leave None to auto-pick from data/recordings.
_RECORDING_MAX_FRAMES = None

from pathlib import Path
import numpy as np

try:
    from ruview.hardware.esp32_capture_analysis import load_esp32_capture
except ImportError:  # Allows the notebook to render in environments without the package installed yet.
    load_esp32_capture = None


def _find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'ruview').exists():
            return candidate
    return current


def _pick_recording_csv(repo_root, override=None):
    if override:
        return Path(override).expanduser().resolve()
    candidates = sorted((repo_root / 'data' / 'recordings').glob('**/*_csi.csv'))
    return candidates[0] if candidates else None


def _elapsed_seconds_for_capture(capture):
    real_ts = np.asarray(capture.timestamps, dtype=np.float64)
    if np.isfinite(real_ts).sum() >= 2 and float(np.nanmax(real_ts) - np.nanmin(real_ts)) > 0:
        return real_ts - float(np.nanmin(real_ts))
    mono = np.asarray(capture.host_monotonic_ns, dtype=np.float64)
    if mono.size == 0:
        return np.array([], dtype=np.float64)
    return (mono - float(mono[0])) / 1_000_000_000.0


def _fill_invalid(values, mask):
    filled = np.asarray(values, dtype=np.float64).copy()
    filled[~mask] = np.nan
    valid = np.isfinite(filled)
    counts = valid.sum(axis=0)
    sums = np.where(valid, filled, 0.0).sum(axis=0)
    col_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    rows, cols = np.where(~valid)
    filled[rows, cols] = col_means[cols]
    return filled


def _capture_to_csi_tensor(capture, max_frames=None):
    take = slice(None) if max_frames is None else slice(0, max_frames)
    amp = _fill_invalid(capture.amplitude[take], capture.valid_mask[take])
    phase = _fill_invalid(capture.phase[take], capture.valid_mask[take])
    time_s = _elapsed_seconds_for_capture(capture)[take]
    csi = (amp * np.exp(1j * phase))[:, None, :]
    return csi, time_s


def _effective_sample_rate(time_s):
    time_s = np.asarray(time_s, dtype=np.float64)
    duration = float(time_s[-1] - time_s[0]) if time_s.size >= 2 else 0.0
    return float((time_s.size - 1) / duration) if duration > 0 else 1.0


_recording_repo_root = _find_repo_root()
_recording_path = _pick_recording_csv(_recording_repo_root, RECORDING_CSV)
_recording_capture = None
_recording_csi = None
_recording_time_s = None

if USE_RECORDING:
    if load_esp32_capture is None:
        raise ImportError('ruview.hardware.esp32_capture_analysis.load_esp32_capture is required for recordings')
    if _recording_path is None:
        raise FileNotFoundError('No *_csi.csv recording found under data/recordings; set RECORDING_CSV explicitly.')
    _recording_capture = load_esp32_capture(_recording_path)
    _recording_csi, _recording_time_s = _capture_to_csi_tensor(_recording_capture, _RECORDING_MAX_FRAMES)
    print(f'Using recording: {_recording_path} ({_recording_csi.shape[0]} frames, {_recording_csi.shape[2]} subcarriers)')
else:
    print('Using synthetic fixture. Set USE_RECORDING=True to plot from a local *_csi.csv recording.')


import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    raise RuntimeError('Install the research extra with: uv sync --extra research') from exc

try:
    from ruview.signal.visualization import subcarrier_heatmap_matrices
except Exception as exc:
    subcarrier_heatmap_matrices = None
    VISUALIZATION_IMPORT_ERROR = exc
else:
    VISUALIZATION_IMPORT_ERROR = None


def inline_heatmap_capture(frames=120, streams=3, subcarriers=64, duration_s=16.0, seed=211):
    rng = np.random.default_rng(seed)
    time_s = np.linspace(0.0, duration_s, frames)
    subcarrier_axis = np.linspace(-1.0, 1.0, subcarriers)
    capture = np.empty((frames, streams, subcarriers), dtype=np.complex128)

    for frame_index, time_value in enumerate(time_s):
        event_envelope = np.exp(-((time_value - 8.0) / 1.6) ** 2)
        event_band = np.exp(-((subcarrier_axis - 0.22) / 0.18) ** 2)
        slow_drift = 0.015 * np.sin(2.0 * np.pi * 0.05 * time_value)

        for stream in range(streams):
            static_amp = 1.0 + 0.05 * np.cos(2.0 * np.pi * (stream + 1) * subcarrier_axis)
            amplitude = static_amp + slow_drift + 0.20 * event_envelope * event_band
            amplitude = amplitude + rng.normal(0.0, 0.007, subcarriers)
            phase = 0.05 * stream + 0.18 * np.sin(np.pi * subcarrier_axis * (stream + 1))
            phase = phase + 0.10 * event_envelope * np.sin(3.0 * np.pi * subcarrier_axis)
            phase = phase + rng.normal(0.0, 0.006, subcarriers)
            capture[frame_index, stream] = amplitude * np.exp(1j * phase)

    return capture, time_s


def rolling_variance(values, window=9):
    half_window = window // 2
    result = np.empty_like(values)

    for index in range(values.shape[0]):
        start = max(0, index - half_window)
        stop = min(values.shape[0], index + half_window + 1)
        result[index] = values[start:stop].var(axis=0)

    return result


if USE_RECORDING and _recording_csi is not None:
    capture = _recording_csi
    time_s = _recording_time_s - float(_recording_time_s[0]) if _recording_time_s.size else _recording_time_s
    source_label = f'recording: {_recording_path}'
else:
    capture, time_s = inline_heatmap_capture()
    source_label = 'inline synthetic fallback'
matrices = {}

if subcarrier_heatmap_matrices is not None:
    try:
        candidate = subcarrier_heatmap_matrices(capture)
        if isinstance(candidate, dict):
            matrices.update(candidate)
    except Exception:
        matrices = {}

amplitude_by_time = matrices.get('amplitude')
phase_by_time = matrices.get('phase')
variance_by_time = matrices.get('rolling_amplitude_variance')

if amplitude_by_time is None:
    amplitude_by_time = np.abs(capture).mean(axis=1)
if phase_by_time is None:
    phase_by_time = np.unwrap(np.angle(capture), axis=2).mean(axis=1)
if variance_by_time is None:
    variance_by_time = rolling_variance(amplitude_by_time, window=9)

summary = {
    'source': source_label,
    'capture_shape': capture.shape,
    'heatmap_shape': amplitude_by_time.shape,
    'max_variance_subcarrier': int(np.argmax(variance_by_time.max(axis=0))),
    'max_variance_time_s': round(float(time_s[np.argmax(variance_by_time.max(axis=1))]), 2),
}

summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
extent = [0, capture.shape[2] - 1, time_s[0], time_s[-1]]

plots = [
    ('Mean amplitude', amplitude_by_time, 'Amplitude', 'viridis'),
    ('Mean unwrapped phase', phase_by_time, 'Phase (radians)', 'twilight'),
    ('Rolling amplitude variance', variance_by_time, 'Variance', 'magma'),
]

for axis, (title, values, colorbar_label, color_map) in zip(axes, plots, strict=True):
    image = axis.imshow(values, aspect='auto', origin='lower', extent=extent, cmap=color_map)
    axis.set_title(title)
    axis.set_xlabel('Subcarrier index')
    axis.set_ylabel('Time (s)')
    fig.colorbar(image, ax=axis, label=colorbar_label)

plt.show()


Expected interpretation: static channel shape should appear as persistent horizontal/vertical structure, while the synthetic motion event should concentrate near the middle of the time axis and around a limited subcarrier band in the variance heatmap.

Limitations: the heatmaps average across streams, use a small centered rolling window, and do not include calibration, missing packets, carrier-frequency offset, or real ESP32 subcarrier indexing. Use the plots as visual debugging fixtures, not as detection thresholds.
